In [ ]:
%pip install networkx matplotlib pydot

In [ ]:
%load_ext autoreload
%autoreload 2

import cpg.cpg_trie_parser as cpg_parser

In [ ]:
from pathlib import Path
import pydot
import networkx as nx

SERVICE_NAME = "frontend"
dot_path = Path(f"services/{SERVICE_NAME}/export.dot")

dot_text = dot_path.read_text(encoding="utf-8")
graphs = pydot.graph_from_dot_data(dot_text)

P = graphs[0]
P

In [ ]:
G = nx.MultiDiGraph()

for node in P.get_nodes():
    name = node.get_name()
    if name in (None, "node", "graph", "edge"):
        continue
    nid = str(name).strip('"')
    attrs = {k: v for k, v in node.get_attributes().items()}
    G.add_node(nid, **attrs)

for edge in P.get_edges():
    src = str(edge.get_source()).strip('"')
    dst = str(edge.get_destination()).strip('"')
    attrs = {k: v for k, v in edge.get_attributes().items()}
    G.add_edge(src, dst, **attrs)

print(G.number_of_nodes(), G.number_of_edges())

In [ ]:
templates = cpg_parser.build_templates_from_cpg(G, max_ddg_depth=5)
root = cpg_parser.build_trie(templates)
cpg_parser.visualize_trie_matplotlib(root, output_path=f"output/{SERVICE_NAME}/trie.png")

In [ ]:
print("Templates:", len(templates))
for t in templates[:10]:
    print(f"[{t.call_node_id}] {t.method_name}() -> {t.raw_template} (static={t.static_count})")

In [ ]:
%pip install pandas

In [ ]:
# Step 1 - Collect all endpoints
from cpg.entrypoint import EntrypointDetector, Entrypoint

print("STEP 1: COLLECTING ALL ENDPOINTS")
print("-" * 40)
detector = EntrypointDetector(G)
all_entrypoints = detector.detect()
print(f"Found {len(all_entrypoints)} entrypoints:")
for i, ep in enumerate(all_entrypoints):
    print(f"  {i+1}. {ep.name} -> {ep.full_name}")

In [ ]:
%pip install graphviz

In [ ]:
def run_stateful_chain_analyzer(
    G,
    templates,
    all_entrypoints,
    service_name: str,
    dataset_path: str,
    start_time: int,
    end_time: int,
    output_dir: str = "output",
    flow_max_depth: int = 5,
    flow_max_paths: int = 50,
    threshold: float = 0.35,
    time_gap_sec: int = 30,
    max_active_chains: int = 2048,
):
    logs = extract_logs(dataset_path, service_name, start_time, end_time)
    df_logs = logs_to_df(logs)

    fsms, summary_df = build_entrypoint_fsms(
        G=G,
        templates=templates,
        all_entrypoints=all_entrypoints,
        service_name=service_name,
        output_dir=output_dir,
        flow_max_depth=flow_max_depth,
        flow_max_paths=flow_max_paths,
    )

    classified_df, active_chains = classify_logs_with_chain_store(
        df_logs,
        fsms,
        threshold=threshold,
        time_gap_sec=time_gap_sec,
        max_active_chains=max_active_chains,
    )

    exports = export_results(classified_df, active_chains, output_dir=output_dir, service_name=service_name)

    print("FSM summary rows:", len(summary_df))
    print("Logs classified:", len(classified_df))
    print("Active chains stored:", len(active_chains))
    print("Exports:", exports)

    return {
        "fsms": fsms,
        "summary_df": summary_df,
        "classified_df": classified_df,
        "active_chains": active_chains,
        "exports": exports,
    }

In [ ]:
from cpg.stateful_chain_store import extract_logs, logs_to_df, build_entrypoint_fsms, classify_logs_with_chain_store
from cpg.visual import export_results
def run_stateful_chain_analyzer(
    G,
    templates,
    all_entrypoints,
    service_name: str,
    dataset_path: str,
    start_time: int,
    end_time: int,
    output_dir: str = "output",
    flow_max_depth: int = 5,
    flow_max_paths: int = 50,
    threshold: float = 0.35,
    time_gap_sec: int = 30,
    max_active_chains: int = 2048,
):
    logs = extract_logs(dataset_path, service_name, start_time, end_time)
    df_logs = logs_to_df(logs)

    fsms, summary_df = build_entrypoint_fsms(
        G=G,
        templates=templates,
        all_entrypoints=all_entrypoints,
        service_name=service_name,
        output_dir=output_dir,
        flow_max_depth=flow_max_depth,
        flow_max_paths=flow_max_paths,
    )

    classified_df, active_chains = classify_logs_with_chain_store(
        df_logs,
        fsms,
        threshold=threshold,
        time_gap_sec=time_gap_sec,
        max_active_chains=max_active_chains,
    )

    exports = export_results(classified_df, active_chains, output_dir=output_dir, service_name=service_name)

    print("FSM summary rows:", len(summary_df))
    print("Logs classified:", len(classified_df))
    print("Active chains stored:", len(active_chains))
    print("Exports:", exports)

    return {
        "fsms": fsms,
        "summary_df": summary_df,
        "classified_df": classified_df,
        "active_chains": active_chains,
        "exports": exports,
    }

In [ ]:
dataset_path="./dataset/re3ob_adservice_f3_1/logs.csv"
start_time=1731903974
end_time=1731903980
logs = extract_logs(dataset_path, SERVICE_NAME, start_time, end_time)
df_logs = logs_to_df(logs)

In [ ]:
import importlib
import cpg.flow
from cpg.flow import EntrypointFlow
importlib.reload(cpg.flow)

entrypoint = next(
    ep for ep in all_entrypoints
    if ep.name == "viewCartHandler"
)

flowanalyzer = EntrypointFlow(
    G,
    max_depth=5,
    max_paths=50,
)

flowresult = flowanalyzer.build(entrypoint.node_id)

In [ ]:
print(flowresult.semantic_graphs)

In [ ]:
from cpg.visual import draw_semantic_graph_graphviz
importlib.reload(cpg.visual)
method_graph = next(
    entry.method_graph
    for entry in flowresult.sequence
    if entry.method_graph.full_name == "main.frontendServer.viewCartHandler"
)

semantic_graph = flowresult.semantic_graphs[
    "main.frontendServer.viewCartHandler"
]

image_path = draw_semantic_graph_graphviz(
    semantic_graph=semantic_graph,
    filename="home_handler_semantic",
    output_dir="output",
    fmt="png",
    rankdir="LR",
)

print(image_path)

In [ ]:
importlib.reload(cpg.visual)
from cpg.visual import visualize_log_fsm
visualize_log_fsm(fsm, output_path=f"output/{SERVICE_NAME}/fsm_{entrypoint.name}.png")

In [ ]:
between_home_and_next = [
    state for state in fsm.states.values()
    if state.previous_log_call_node_id and state.kind == "BETWEEN_LOGS"
]

for state in between_home_and_next[:10]:
    print(state.id, "conditions=", state.conditions, "methods=", state.direct_methods)

In [ ]:
for segment_id in ["segment:5", "segment:9"]:
    state = fsm.states[segment_id]
    print(segment_id, "external=", state.external_calls)
    print(segment_id, "prev_log=", state.previous_log_call_node_id)
    print(segment_id, "next_log=", state.next_log_call_node_id)

In [ ]:
methods_with_own_logs = {
    transition.method_full_name
    for transition in fsm.transitions
}

print("Методы с собственным логом:")
print(*sorted(methods_with_own_logs), sep="\n")

In [ ]:
threshold: float = 0.35
time_gap_sec: int = 30
max_active_chains: int = 2048

classified_df, active_chains = classify_logs_with_chain_store(
    df_logs,
    fsms,
    threshold=threshold,
    time_gap_sec=time_gap_sec,
    max_active_chains=max_active_chains,
)

exports = export_results(classified_df, active_chains, output_dir="output", service_name=SERVICE_NAME)

print("FSM summary rows:", len(summary_df))
print("Logs classified:", len(classified_df))
print("Active chains stored:", len(active_chains))
print("Exports:", exports)

In [ ]:
print(active_chains[1])